# DevBugger Model Training and Validation

Compare five classifiers and five regressors using five-fold cross-validation on the training reports in `devbugger_split.csv`. Select classifiers by weighted F1 and regressors by RMSE in hours. Run the cells in order from the project folder using the project Python environment.

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

SEED = 42

In [2]:
reports = pd.read_csv("devbugger_dataset_800.csv").set_index("row_id")
split = pd.read_csv("devbugger_split.csv").set_index("row_id")
assert reports.index.is_unique and split.index.is_unique
assert split.index.sort_values().equals(reports.index.sort_values())
assert split["split"].isin(["train", "test"]).all()

train_ids = split.index[split["split"].eq("train")]
train = reports.loc[train_ids]
numeric_columns = ["title_len_words", "body_len_chars"]
X_train = train[["source", *numeric_columns]].copy()
X_train["report_text"] = train["report_title"].fillna("") + " " + train["body"].fillna("")
y_class = train["Bug_Type"]
y_hours = train["resolution_hours"]
X_reg = X_train.assign(Bug_Type=y_class)

class_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
reg_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
reg_scoring = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"}
print(f"Training reports: {len(train)}")

Training reports: 640


## Fold-specific preprocessing

Each pipeline learns its vocabulary, category encoding and scaling values from the training portion of each fold. Classification uses text, source and two length measurements. The known-category regression experiment adds the recorded bug type. Learned regressors fit log-transformed durations and convert predictions back to hours for scoring.

In [3]:
def make_model_pipeline(model, categories):
    features = ColumnTransformer([
        ("text", TfidfVectorizer(ngram_range=(1, 2), min_df=2), "report_text"),
        ("categories", OneHotEncoder(handle_unknown="ignore"), categories),
        ("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_columns),
    ])
    return Pipeline([("features", features), ("model", model)])


def make_regression_pipeline(model):
    return TransformedTargetRegressor(
        regressor=make_model_pipeline(model, ["source", "Bug_Type"]),
        func=np.log1p,
        inverse_func=np.expm1,
    )

## Simple benchmarks

The classification benchmark always predicts the most frequent training class. The regression benchmark always predicts the arithmetic mean of the training durations in hours. During cross-validation each benchmark learns its constant again from the training rows of that fold.

In [4]:
counts = y_class.value_counts()
mean_hours = y_hours.mean()
errors = y_hours - mean_hours
print(f"Majority class: {counts.idxmax()} | training accuracy: {counts.max() / len(train):.2%}")
print(f"Mean-time prediction: {mean_hours:,.2f} hours")
print(f"Training MAE: {errors.abs().mean():,.2f} | RMSE: {np.sqrt(errors.pow(2).mean()):,.2f}")

dummy_class = cross_validate(
    DummyClassifier(strategy="most_frequent"), X_train, y_class,
    cv=class_cv, scoring=["accuracy", "f1_weighted"],
)
default_logistic = cross_validate(
    make_model_pipeline(LogisticRegression(class_weight="balanced", max_iter=1000), ["source"]),
    X_train, y_class, cv=class_cv, scoring="accuracy",
)
baseline_cv = cross_validate(
    DummyRegressor(strategy="mean"), X_reg, y_hours, cv=reg_cv, scoring=reg_scoring,
)
print(f"Majority-class validation accuracy: {dummy_class['test_accuracy'].mean():.2%}")
print(f"Majority-class validation weighted F1: {dummy_class['test_f1_weighted'].mean():.4f}")
print(f"Default logistic validation accuracy: {default_logistic['test_score'].mean():.2%}")

Majority class: CORS_Error | training accuracy: 20.78%
Mean-time prediction: 3,296.44 hours
Training MAE: 4,816.92 | RMSE: 8,827.80
Majority-class validation accuracy: 20.78%
Majority-class validation weighted F1: 0.0715
Default logistic validation accuracy: 91.88%


## Classification model comparison

Each candidate uses the same stratified folds. The table gives the mean weighted F1 and its standard deviation across folds at each model's selected settings.

In [5]:
class_candidates = {
    "Logistic Regression": (
        LogisticRegression(class_weight="balanced", max_iter=1000),
        {"model__C": [0.1, 1.0, 10.0]},
    ),
    "KNN": (KNeighborsClassifier(), {
        "model__n_neighbors": [3, 5, 9], "model__weights": ["uniform", "distance"],
    }),
    "Decision Tree": (DecisionTreeClassifier(class_weight="balanced", random_state=SEED), {
        "model__max_depth": [5, 15, None], "model__min_samples_split": [2, 10],
    }),
    "Random Forest": (RandomForestClassifier(class_weight="balanced", random_state=SEED, n_jobs=-1), {
        "model__n_estimators": [100, 200], "model__max_depth": [15, 30, None],
    }),
    "SVM": (SVC(class_weight="balanced"), {
        "model__kernel": ["linear", "rbf"], "model__C": [0.1, 1.0, 10.0],
    }),
}

class_searches = {}
class_rows = []
for name, (model, grid) in class_candidates.items():
    search = GridSearchCV(
        make_model_pipeline(model, ["source"]), grid, cv=class_cv, scoring="f1_weighted",
    )
    search.fit(X_train, y_class)
    class_searches[name] = search
    class_rows.append({
        "Model": name,
        "Mean weighted F1": search.best_score_,
        "Fold SD": search.cv_results_["std_test_score"][search.best_index_],
    })
    print(name, search.best_params_)

classification_results = pd.DataFrame(class_rows).sort_values("Mean weighted F1", ascending=False)
print(classification_results.round(4).to_string(index=False))

Logistic Regression {'model__C': 10.0}
KNN {'model__n_neighbors': 5, 'model__weights': 'distance'}
Decision Tree {'model__max_depth': 15, 'model__min_samples_split': 10}
Random Forest {'model__max_depth': 30, 'model__n_estimators': 200}
SVM {'model__C': 1.0, 'model__kernel': 'linear'}
              Model  Mean weighted F1  Fold SD
Logistic Regression            0.9359   0.0208
                SVM            0.9259   0.0249
      Random Forest            0.9179   0.0204
      Decision Tree            0.8004   0.0271
                KNN            0.7351   0.0148


## Known-category regression comparison

Linear Regression uses default settings. The other four regressors use grid search with the same shuffled regression folds. Scikit-learn negates error scores for ranking. The table restores positive values and reports the mean of the fold-level errors in hours.

In [6]:
linear_cv = cross_validate(
    make_regression_pipeline(LinearRegression()), X_reg, y_hours, cv=reg_cv, scoring=reg_scoring,
)
reg_rows = [
    ("Mean-time baseline", -baseline_cv["test_mae"].mean(), -baseline_cv["test_rmse"].mean()),
    ("Linear Regression", -linear_cv["test_mae"].mean(), -linear_cv["test_rmse"].mean()),
]

reg_candidates = {
    "KNN": (KNeighborsRegressor(), {"regressor__model__n_neighbors": [3, 5, 9]}),
    "Decision Tree": (DecisionTreeRegressor(random_state=SEED), {
        "regressor__model__max_depth": [5, 15, None],
        "regressor__model__min_samples_leaf": [1, 5, 10],
    }),
    "Random Forest": (RandomForestRegressor(max_features="sqrt", random_state=SEED, n_jobs=-1), {
        "regressor__model__n_estimators": [100, 200],
        "regressor__model__max_depth": [15, 30, None],
    }),
    "SVR": (SVR(kernel="rbf"), {
        "regressor__model__C": [0.1, 1.0, 10.0],
        "regressor__model__epsilon": [0.1, 0.5],
    }),
}

reg_searches = {}
for name, (model, grid) in reg_candidates.items():
    search = GridSearchCV(
        make_regression_pipeline(model), grid, cv=reg_cv, scoring=reg_scoring, refit="rmse",
    )
    search.fit(X_reg, y_hours)
    reg_searches[name] = search
    mae = -search.cv_results_["mean_test_mae"][search.best_index_]
    reg_rows.append((name, mae, -search.best_score_))
    print(name, search.best_params_)

regression_results = pd.DataFrame(reg_rows, columns=["Model", "MAE (hours)", "RMSE (hours)"])
print(regression_results.sort_values("RMSE (hours)").round(2).to_string(index=False))

KNN {'regressor__model__n_neighbors': 3}
Decision Tree {'regressor__model__max_depth': None, 'regressor__model__min_samples_leaf': 10}
Random Forest {'regressor__model__max_depth': None, 'regressor__model__n_estimators': 200}
SVR {'regressor__model__C': 10.0, 'regressor__model__epsilon': 0.1}
             Model  MAE (hours)  RMSE (hours)
Mean-time baseline      4821.83       8775.46
 Linear Regression      3233.64       8954.21
               KNN      3424.48       8964.07
               SVR      3252.98       9113.78
     Decision Tree      3574.18       9116.72
     Random Forest      3251.45       9282.77


## Interpretation

Logistic Regression had the highest validation weighted F1 (0.9359), followed by the linear SVM (0.9259). Linear Regression had the lowest validation MAE and RMSE among the learned regressors. Its MAE was about 1,588 hours lower than the mean-time baseline, while its RMSE was about 179 hours higher. Final evaluation will measure performance on the held-out reports, including the resolution-time workflow that receives a predicted bug category.